In [1]:
# ============================================================
# ALPHABET.CSV - NLP + TF-IDF + K-MEANS CLUSTERING
# Complete Jupyter Notebook Code
# ============================================================


# ============================================================
# CELL 1: INSTALL LIBRARIES
# ============================================================

# Run this cell if the libraries are not already installed.

# !pip install pandas numpy matplotlib seaborn scikit-learn nltk


# ============================================================
# CELL 2: IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
import re
import string
import warnings

warnings.filterwarnings("ignore")

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score


# ============================================================
# CELL 3: DOWNLOAD NLTK DATA
# ============================================================

nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

print("NLTK resources downloaded successfully.")


# ============================================================
# CELL 4: LOAD DATASET
# ============================================================

# Make sure alphabet.csv is in the same folder as this notebook.

file_name = "alphabet.csv"

df = pd.read_csv(file_name)

print("Dataset loaded successfully!")
print()
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())


# ============================================================
# CELL 5: DATASET INFORMATION
# ============================================================

print("========== DATASET INFORMATION ==========\n")

print(df.info())

print("\n========== MISSING VALUES ==========\n")

print(df.isnull().sum())

print("\n========== DUPLICATE ROWS ==========\n")

print("Number of duplicate rows:", df.duplicated().sum())


# ============================================================
# CELL 6: REMOVE DUPLICATES
# ============================================================

df = df.drop_duplicates().reset_index(drop=True)

print("Dataset shape after removing duplicates:", df.shape)


# ============================================================
# CELL 7: FIND TEXT COLUMN
# ============================================================

# The code tries to find a text column automatically.

possible_text_columns = [
    "text",
    "Text",
    "TEXT",
    "description",
    "Description",
    "sentence",
    "Sentence",
    "content",
    "Content",
    "word",
    "Word",
    "name",
    "Name",
    "label",
    "Label"
]

text_column = None

# First, look for common text-column names
for column in possible_text_columns:
    if column in df.columns:
        text_column = column
        break

# If no common text column exists,
# find the first column containing strings.
if text_column is None:

    object_columns = df.select_dtypes(
        include=["object", "string"]
    ).columns

    if len(object_columns) > 0:
        text_column = object_columns[0]

# Stop if no text column exists
if text_column is None:
    raise ValueError(
        "No text column was found in alphabet.csv. "
        "Please specify the text column manually."
    )

print("Selected text column:", text_column)


# ============================================================
# CELL 8: PREPARE TEXT DATA
# ============================================================

df[text_column] = df[text_column].fillna("")
df[text_column] = df[text_column].astype(str)

print("Text data prepared successfully.")


# ============================================================
# CELL 9: INITIALIZE NLP TOOLS
# ============================================================

stop_words = set(stopwords.words("english"))

lemmatizer = WordNetLemmatizer()


# ============================================================
# CELL 10: NLP TEXT CLEANING FUNCTION
# ============================================================

def clean_text(text):

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(
        r"http\S+|www\S+|https\S+",
        "",
        text
    )

    # Remove email addresses
    text = re.sub(
        r"\S+@\S+",
        "",
        text
    )

    # Remove numbers
    text = re.sub(
        r"\d+",
        "",
        text
    )

    # Remove punctuation
    text = text.translate(
        str.maketrans(
            "",
            "",
            string.punctuation
        )
    )

    # Remove non-English/non-alphabet characters
    text = re.sub(
        r"[^a-zA-Z\s]",
        " ",
        text
    )

    # Remove extra spaces
    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    # Split into words
    words = text.split()

    # Remove stopwords
    words = [
        word
        for word in words
        if word not in stop_words
    ]

    # Lemmatization
    words = [
        lemmatizer.lemmatize(word)
        for word in words
    ]

    # Remove very short words
    words = [
        word
        for word in words
        if len(word) > 1
    ]

    # Return cleaned sentence
    return " ".join(words)


# ============================================================
# CELL 11: APPLY NLP CLEANING
# ============================================================

df["clean_text"] = df[text_column].apply(clean_text)

print("Original text vs cleaned text:\n")

display(
    df[
        [text_column, "clean_text"]
    ].head(15)
)


# ============================================================
# CELL 12: REMOVE EMPTY TEXT
# ============================================================

df = df[
    df["clean_text"].str.strip() != ""
].copy()

df = df.reset_index(drop=True)

print("Rows remaining after text cleaning:", len(df))


# ============================================================
# CELL 13: TEXT STATISTICS
# ============================================================

df["word_count"] = df["clean_text"].apply(
    lambda x: len(x.split())
)

print("Average number of words:")

print(
    round(
        df["word_count"].mean(),
        2
    )
)

print("\nMinimum words:")

print(df["word_count"].min())

print("\nMaximum words:")

print(df["word_count"].max())


# ============================================================
# CELL 14: WORD COUNT DISTRIBUTION
# ============================================================

plt.figure(figsize=(10, 6))

plt.hist(
    df["word_count"],
    bins=20
)

plt.xlabel("Number of Words")
plt.ylabel("Frequency")

plt.title(
    "Distribution of Words per Text Record"
)

plt.show()


# ============================================================
# CELL 15: CREATE TF-IDF FEATURES
# ============================================================

tfidf = TfidfVectorizer(

    # Maximum number of features
    max_features=5000,

    # Use single words and two-word combinations
    ngram_range=(1, 2),

    # Ignore extremely rare terms
    min_df=1,

    # Ignore extremely common terms
    max_df=0.95,

    # Remove English stopwords again at vectorization level
    stop_words="english"
)


# Transform text into TF-IDF matrix
X = tfidf.fit_transform(
    df["clean_text"]
)

print("TF-IDF completed.")

print(
    "TF-IDF matrix shape:",
    X.shape
)


# ============================================================
# CELL 16: DISPLAY TF-IDF FEATURES
# ============================================================

feature_names = tfidf.get_feature_names_out()

print(
    "Number of TF-IDF features:",
    len(feature_names)
)

print("\nFirst 50 features:")

print(
    feature_names[:50]
)


# ============================================================
# CELL 17: ELBOW METHOD
# ============================================================

# Determine maximum possible K
if len(df) < 3:

    raise ValueError(
        "The dataset needs at least 3 records for clustering."
    )

max_k = min(
    10,
    len(df) - 1
)

k_values = list(
    range(2, max_k + 1)
)

inertia_values = []


for k in k_values:

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    model.fit(X)

    inertia_values.append(
        model.inertia_
    )


# Plot elbow curve
plt.figure(figsize=(10, 6))

plt.plot(
    k_values,
    inertia_values,
    marker="o"
)

plt.xlabel(
    "Number of Clusters (K)"
)

plt.ylabel(
    "Inertia"
)

plt.title(
    "Elbow Method for Choosing K"
)

plt.xticks(k_values)

plt.grid(True)

plt.show()


# ============================================================
# CELL 18: SILHOUETTE SCORE
# ============================================================

silhouette_values = []


for k in k_values:

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = model.fit_predict(X)

    score = silhouette_score(
        X,
        labels
    )

    silhouette_values.append(
        score
    )

    print(
        f"K = {k} | "
        f"Silhouette Score = {score:.4f}"
    )


# ============================================================
# CELL 19: PLOT SILHOUETTE SCORES
# ============================================================

plt.figure(figsize=(10, 6))

plt.plot(
    k_values,
    silhouette_values,
    marker="o"
)

plt.xlabel(
    "Number of Clusters (K)"
)

plt.ylabel(
    "Silhouette Score"
)

plt.title(
    "Silhouette Score for Different K Values"
)

plt.xticks(k_values)

plt.grid(True)

plt.show()


# ============================================================
# CELL 20: SELECT BEST K
# ============================================================

best_index = np.argmax(
    silhouette_values
)

best_k = k_values[best_index]

best_score = silhouette_values[best_index]

print(
    "Best K:",
    best_k
)

print(
    "Best Silhouette Score:",
    round(best_score, 4)
)


# ============================================================
# CELL 21: TRAIN FINAL K-MEANS MODEL
# ============================================================

kmeans = KMeans(

    n_clusters=best_k,

    random_state=42,

    n_init=10
)


# Fit model and predict clusters
df["cluster"] = kmeans.fit_predict(X)

print(
    "K-Means clustering completed successfully."
)


# ============================================================
# CELL 22: CLUSTER COUNTS
# ============================================================

cluster_counts = (
    df["cluster"]
    .value_counts()
    .sort_index()
)

print(
    "Number of records in each cluster:\n"
)

print(cluster_counts)


# ============================================================
# CELL 23: CLUSTER DISTRIBUTION GRAPH
# ============================================================

plt.figure(figsize=(10, 6))

sns.barplot(
    x=cluster_counts.index,
    y=cluster_counts.values
)

plt.xlabel(
    "Cluster"
)

plt.ylabel(
    "Number of Records"
)

plt.title(
    "Number of Records in Each Cluster"
)

plt.show()


# ============================================================
# CELL 24: IDENTIFY IMPORTANT WORDS
# ============================================================

print(
    "\n========== IMPORTANT WORDS BY CLUSTER ==========\n"
)


for cluster_number in range(best_k):

    # Get cluster center
    center = kmeans.cluster_centers_[
        cluster_number
    ]

    # Sort features by importance
    top_indices = center.argsort()[
        ::-1
    ][:15]

    top_words = [
        feature_names[index]
        for index in top_indices
    ]

    print(
        f"Cluster {cluster_number}:"
    )

    print(
        ", ".join(top_words)
    )

    print()


# ============================================================
# CELL 25: SHOW EXAMPLES FROM EACH CLUSTER
# ============================================================

for cluster_number in range(best_k):

    print()
    print("=" * 70)

    print(
        f"CLUSTER {cluster_number}"
    )

    print("=" * 70)

    cluster_data = df[
        df["cluster"] == cluster_number
    ]

    examples = cluster_data[
        [text_column, "clean_text"]
    ].head(10)

    display(examples)


# ============================================================
# CELL 26: PCA DIMENSIONALITY REDUCTION
# ============================================================

# PCA requires dense data.
# For very large datasets, TruncatedSVD is preferable.

X_dense = X.toarray()

pca = PCA(
    n_components=2,
    random_state=42
)

X_pca = pca.fit_transform(
    X_dense
)

print(
    "PCA completed."
)

print(
    "Explained variance ratio:",
    pca.explained_variance_ratio_
)

print(
    "Total explained variance:",
    round(
        sum(
            pca.explained_variance_ratio_
        ) * 100,
        2
    ),
    "%"
)


# ============================================================
# CELL 27: CREATE PCA DATAFRAME
# ============================================================

pca_df = pd.DataFrame({

    "PCA1": X_pca[:, 0],

    "PCA2": X_pca[:, 1],

    "Cluster": df["cluster"].values

})


# ============================================================
# CELL 28: VISUALIZE K-MEANS CLUSTERS
# ============================================================

plt.figure(figsize=(12, 8))

sns.scatterplot(

    data=pca_df,

    x="PCA1",

    y="PCA2",

    hue="Cluster",

    palette="tab10",

    s=100,

    alpha=0.8
)

plt.title(
    "K-Means Clustering Using NLP + TF-IDF"
)

plt.xlabel(
    "PCA Component 1"
)

plt.ylabel(
    "PCA Component 2"
)

plt.legend(
    title="Cluster"
)

plt.grid(True)

plt.show()


# ============================================================
# CELL 29: SHOW CLUSTER CENTERS
# ============================================================

print(
    "Cluster centers shape:",
    kmeans.cluster_centers_.shape
)


# ============================================================
# CELL 30: MOST IMPORTANT WORDS FOR EACH CLUSTER
# ============================================================

cluster_words = {}


for cluster_number in range(best_k):

    center = kmeans.cluster_centers_[
        cluster_number
    ]

    top_indices = center.argsort()[
        ::-1
    ][:20]

    words = [
        feature_names[i]
        for i in top_indices
    ]

    cluster_words[
        cluster_number
    ] = words


# Create table
cluster_word_table = pd.DataFrame(
    dict(
        (
            cluster,
            pd.Series(words)
        )
        for cluster, words
        in cluster_words.items()
    )
)

cluster_word_table.index = [
    f"Top Word {i+1}"
    for i in range(
        len(cluster_word_table)
    )
]

display(
    cluster_word_table
)


# ============================================================
# CELL 31: WORD FREQUENCY ANALYSIS
# ============================================================

from collections import Counter

all_words = []

for text in df["clean_text"]:

    all_words.extend(
        text.split()
    )

word_frequency = Counter(
    all_words
)

most_common_words = (
    word_frequency
    .most_common(20)
)

print(
    "Most common words:"
)

for word, count in most_common_words:

    print(
        f"{word}: {count}"
    )


# ============================================================
# CELL 32: MOST COMMON WORDS GRAPH
# ============================================================

words = [
    item[0]
    for item in most_common_words
]

counts = [
    item[1]
    for item in most_common_words
]

plt.figure(figsize=(12, 7))

sns.barplot(
    x=counts,
    y=words
)

plt.xlabel(
    "Frequency"
)

plt.ylabel(
    "Word"
)

plt.title(
    "Top 20 Most Common Words"
)

plt.show()


# ============================================================
# CELL 33: SAVE CLUSTERED DATASET
# ============================================================

output_file = "alphabet_clustered.csv"

df.to_csv(
    output_file,
    index=False
)

print(
    f"Clustered dataset saved as: {output_file}"
)


# ============================================================
# CELL 34: SAVE CLUSTER SUMMARY
# ============================================================

summary_data = []

for cluster_number in range(best_k):

    cluster_size = (
        df["cluster"] == cluster_number
    ).sum()

    words = ", ".join(
        cluster_words[
            cluster_number
        ][:15]
    )

    summary_data.append({

        "Cluster": cluster_number,

        "Number_of_Records": cluster_size,

        "Important_Words": words

    })


cluster_summary = pd.DataFrame(
    summary_data
)

display(
    cluster_summary
)

cluster_summary.to_csv(
    "alphabet_cluster_summary.csv",
    index=False
)

print(
    "Cluster summary saved as "
    "alphabet_cluster_summary.csv"
)


# ============================================================
# CELL 35: FINAL RESULTS
# ============================================================

print("\n")
print("=" * 70)
print("FINAL RESULTS")
print("=" * 70)

print(
    "\nOriginal dataset shape:",
    df.shape
)

print(
    "Text column:",
    text_column
)

print(
    "TF-IDF features:",
    X.shape[1]
)

print(
    "Selected number of clusters:",
    best_k
)

print(
    "Silhouette score:",
    round(best_score, 4)
)

print(
    "\nCluster sizes:"
)

print(
    df["cluster"]
    .value_counts()
    .sort_index()
)

print("\nOutput files:")
print("1. alphabet_clustered.csv")
print("2. alphabet_cluster_summary.csv")

print(
    "\nK-Means NLP analysis completed successfully!"
)

NLTK resources downloaded successfully.
Dataset loaded successfully!

Number of rows: 26
Number of columns: 2

Column names:
['letter', 'frequency']

First 5 rows:


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


,letter,frequency
0,E,0.12702
1,T,0.09056
2,A,0.08167
3,O,0.07507
4,I,0.06966


========== DATASET INFORMATION ==========

<class 'pandas.DataFrame'>
RangeIndex: 26 entries, 0 to 25
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   letter     26 non-null     str    
 1   frequency  26 non-null     float64
dtypes: float64(1), str(1)
memory usage: 574.0 bytes
None

========== MISSING VALUES ==========

letter       0
frequency    0
dtype: int64

========== DUPLICATE ROWS ==========

Number of duplicate rows: 0
Dataset shape after removing duplicates: (26, 2)
Selected text column: letter
Text data prepared successfully.
Original text vs cleaned text:



,letter,clean_text
0,E,
1,T,
2,A,
3,O,
4,I,
5,N,
6,S,
7,H,
8,R,
9,D,


Rows remaining after text cleaning: 0
Average number of words:


TypeError: Cannot perform reduction 'mean' with string dtype